In [1]:
!pip install -q torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 40.4 MB/s eta 0:00:00


In [3]:
import torch, random, copy
import numpy as np, pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score
import torch_geometric.transforms as T
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

dataset = Planetoid(root='data/Planetoid', name='Cora', transform=T.NormalizeFeatures())
data = dataset[0]
x = data.x.to(device)
N = data.num_nodes

Processing...
Done!


##Split unique positive links

In [5]:
ei = data.edge_index
unique_pos = ei[:, ei[0] < ei[1]]                 # each undirected link once
M = unique_pos.size(1)
print("Unique undirected links:", M)

g = torch.Generator().manual_seed(0)
unique_pos = unique_pos[:, torch.randperm(M, generator=g)]

n_train, n_val = int(0.7 * M), int(0.1 * M)       # 70% / 10% / 20%
train_pos = unique_pos[:, :n_train]
val_pos   = unique_pos[:, n_train:n_train + n_val]
test_pos  = unique_pos[:, n_train + n_val:]
print("train/val/test positives:", train_pos.size(1), val_pos.size(1), test_pos.size(1))

def both_directions(pos):
    return torch.cat([pos, pos.flip(0)], dim=1)

train_mp = both_directions(train_pos).to(device)   # the ONLY graph used for message passing

Unique undirected links: 5278
train/val/test positives: 3694 527 1057


##Negative pairs

In [ ]:
##Negative pairs

In [6]:
all_pos_set = set(map(tuple, unique_pos.t().tolist()))     # every observed link, as (u, v) with u < v

rng = np.random.default_rng(0)
neg_list, neg_set = [], set()
while len(neg_list) < M:                                    # one negative per positive
    u, v = int(rng.integers(0, N)), int(rng.integers(0, N))
    if u == v:
        continue                                            # no self-links
    pair = (min(u, v), max(u, v))
    if pair in all_pos_set or pair in neg_set:
        continue                                            # not an observed link, no duplicates
    neg_set.add(pair); neg_list.append(pair)

neg = torch.tensor(neg_list).t()
train_neg = neg[:, :n_train]
val_neg   = neg[:, n_train:n_train + n_val]
test_neg  = neg[:, n_train + n_val:]
print("train/val/test negatives:", train_neg.size(1), val_neg.size(1), test_neg.size(1))

train/val/test negatives: 3694 527 1057


##Automated leakage checks

In [7]:
def pair_set(p):
    return set(map(tuple, p.t().tolist()))

def edge_set(edge_index):
    return set(map(tuple, edge_index.cpu().t().tolist()))

def check_splits():
    tr, va, te = pair_set(train_pos), pair_set(val_pos), pair_set(test_pos)
    assert not (tr & va) and not (tr & te) and not (va & te), "positive splits overlap"
    assert len(tr | va | te) == M == len(all_pos_set), "positives lost or duplicated"
    ntr, nva, nte = pair_set(train_neg), pair_set(val_neg), pair_set(test_neg)
    assert not (ntr & nva) and not (ntr & nte) and not (nva & nte), "negative splits overlap"
    for n in (ntr, nva, nte):
        assert not (n & all_pos_set), "a negative is an observed link"
        assert all(u != v for u, v in n), "self-link in negatives"
    print("Split checks passed")

def check_no_leakage(mp_edge_index):
    """Held-out links AND their reverse copies must not be in the message-passing graph."""
    mp = edge_set(mp_edge_index)
    assert mp == {(v, u) for u, v in mp}, "message-passing graph is not symmetric"
    for name, pos in [('val', val_pos), ('test', test_pos)]:
        for u, v in pos.t().tolist():
            assert (u, v) not in mp, f"{name} link {(u, v)} is in the message-passing graph"
            assert (v, u) not in mp, f"{name} reverse link {(v, u)} is in the message-passing graph"
    assert len(mp) == 2 * train_pos.size(1), "graph has more edges than the training links"
    print("No-leakage checks passed")

check_splits()
check_no_leakage(train_mp)

Split checks passed
No-leakage checks passed


In [8]:
##Models (GCN encoder, two decoders)

In [9]:
class Encoder(nn.Module):
    def __init__(self, in_dim, hidden=128, out_dim=64):
        super().__init__()
        self.c1 = GCNConv(in_dim, hidden)
        self.c2 = GCNConv(hidden, out_dim)
    def forward(self, x, edge_index):
        x = F.dropout(x, 0.5, self.training)
        x = F.relu(self.c1(x, edge_index))
        x = F.dropout(x, 0.5, self.training)
        return self.c2(x, edge_index)

class DotDecoder(nn.Module):
    def forward(self, za, zb):
        return (za * zb).sum(-1)

class MLPDecoder(nn.Module):
    # features z_a*z_b and |z_a - z_b| are both unchanged if a and b are swapped
    def __init__(self, d=64, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2 * d, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, za, zb):
        feats = torch.cat([za * zb, (za - zb).abs()], dim=-1)
        return self.net(feats).squeeze(-1)

class LinkModel(nn.Module):
    def __init__(self, in_dim, decoder):
        super().__init__()
        self.encoder = Encoder(in_dim)
        self.decoder = DotDecoder() if decoder == 'dot' else MLPDecoder()
    def score(self, z, pairs):
        return self.decoder(z[pairs[0]], z[pairs[1]])

In [10]:
test_model = LinkModel(x.size(1), 'mlp').to(device).eval()
with torch.no_grad():
    z = test_model.encoder(x, train_mp)
    p = test_pos.to(device)
    assert torch.allclose(test_model.score(z, p), test_model.score(z, p.flip(0))), "decoder is not symmetric"
print("MLP decoder symmetry check passed")

MLP decoder symmetry check passed


Training and evaluation

In [11]:
def evaluate(model, mp_edge_index, pos, neg):
    model.eval()
    with torch.no_grad():
        z = model.encoder(x, mp_edge_index)
        s = torch.cat([model.score(z, pos.to(device)), model.score(z, neg.to(device))]).cpu().numpy()
    y = np.r_[np.ones(pos.size(1)), np.zeros(neg.size(1))]
    return roc_auc_score(y, s), average_precision_score(y, s)

def train_link(decoder, mp_edge_index, seed, use_val=True, epochs=200, lr=0.01):
    set_seed(seed)
    model = LinkModel(x.size(1), decoder).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    tp, tn = train_pos.to(device), train_neg.to(device)
    labels = torch.cat([torch.ones(tp.size(1)), torch.zeros(tn.size(1))]).to(device)
    best_auc, best_state = -1, None
    for epoch in range(epochs):
        model.train(); opt.zero_grad()
        z = model.encoder(x, mp_edge_index)
        logits = torch.cat([model.score(z, tp), model.score(z, tn)])
        loss = F.binary_cross_entropy_with_logits(logits, labels)
        loss.backward(); opt.step()
        if use_val:                                   # checkpoint chosen on validation only
            val_auc, _ = evaluate(model, mp_edge_index, val_pos, val_neg)
            if val_auc > best_auc:
                best_auc, best_state = val_auc, copy.deepcopy(model.state_dict())
    if use_val:
        model.load_state_dict(best_state)
    return model

##Common-neighbor baseline (training links only)

In [12]:
def common_neighbor_matrix(edge_index):
    A = torch.zeros(N, N)
    A[edge_index[0].cpu(), edge_index[1].cpu()] = 1
    return A @ A                                      # entry [u, v] = number of shared neighbours

def eval_cn(CN, pos, neg):
    s = np.r_[CN[pos[0], pos[1]].numpy(), CN[neg[0], neg[1]].numpy()]
    y = np.r_[np.ones(pos.size(1)), np.zeros(neg.size(1))]
    return roc_auc_score(y, s), average_precision_score(y, s)

CN_train = common_neighbor_matrix(train_mp)
print("CN baseline, valid test AUC/AP:", eval_cn(CN_train, test_pos, test_neg))

CN baseline, valid test AUC/AP: (np.float64(0.6639683723144975), np.float64(0.6622358877325762))


##Valid experiment

In [14]:
rows, models = [], {}
for seed in [0, 1, 2]:
    for dec in ['dot', 'mlp']:
        m = train_link(dec, train_mp, seed, use_val=True)
        val_auc, _ = evaluate(m, train_mp, val_pos, val_neg)
        auc, ap = evaluate(m, train_mp, test_pos, test_neg)
        rows.append(dict(setting='valid', method=f'GCN + {dec}', seed=seed, val_auc=val_auc, test_auc=auc, test_ap=ap))
        if seed == 0:
            models[dec] = m

auc, ap = eval_cn(CN_train, test_pos, test_neg)
rows.append(dict(setting='valid', method='Common neighbours', seed=0, val_auc=eval_cn(CN_train, val_pos, val_neg)[0], test_auc=auc, test_ap=ap))

valid_df = pd.DataFrame(rows)
print(valid_df.groupby('method')[['val_auc', 'test_auc', 'test_ap']].agg(['mean', 'std']).round(4))

                  val_auc         test_auc         test_ap        
                     mean     std     mean     std    mean     std
method                                                            
Common neighbours  0.6661     NaN   0.6640     NaN  0.6622     NaN
GCN + dot          0.7569  0.0013   0.7609  0.0251  0.7581  0.0180
GCN + mlp          0.8494  0.0338   0.8617  0.0277  0.8537  0.0292


In [15]:
best_dec = valid_df[valid_df.method.str.startswith('GCN')].groupby('method').val_auc.mean().idxmax()
best_dec = best_dec.split(' + ')[1]
print("Best decoder by validation:", best_dec)

Best decoder by validation: mlp


In [16]:
leaky_mp = both_directions(unique_pos).to(device)      # train + val + test links all visible

try:
    check_no_leakage(leaky_mp)
except AssertionError as e:
    print("Leakage check correctly FAILED on the leaky graph:", e)

leaky_rows = []
for seed in [0, 1, 2]:
    for dec in ['dot', 'mlp']:
        m = train_link(dec, leaky_mp, seed, use_val=False)
        auc, ap = evaluate(m, leaky_mp, test_pos, test_neg)
        leaky_rows.append(dict(setting='LEAKY', method=f'GCN + {dec}', seed=seed, test_auc=auc, test_ap=ap))

CN_leaky = common_neighbor_matrix(leaky_mp)
auc, ap = eval_cn(CN_leaky, test_pos, test_neg)
leaky_rows.append(dict(setting='LEAKY', method='Common neighbours', seed=0, test_auc=auc, test_ap=ap))

leaky_df = pd.DataFrame(leaky_rows)
all_df = pd.concat([valid_df, leaky_df])
comparison = all_df.groupby(['method', 'setting'])[['test_auc', 'test_ap']].agg(['mean', 'std']).round(4)
print(comparison)

Leakage check correctly FAILED on the leaky graph: val link (306, 519) is in the message-passing graph
                          test_auc         test_ap        
                              mean     std    mean     std
method            setting                                 
Common neighbours LEAKY     0.7649     NaN  0.7629     NaN
                  valid     0.6640     NaN  0.6622     NaN
GCN + dot         LEAKY     0.8158  0.0111  0.8118  0.0076
                  valid     0.7609  0.0251  0.7581  0.0180
GCN + mlp         LEAKY     0.9561  0.0175  0.9485  0.0169
                  valid     0.8617  0.0277  0.8537  0.0292


##Failure cases

In [18]:
model = models[best_dec]
deg = torch.bincount(train_mp[0].cpu(), minlength=N)       # degree in the training graph
y_cls = data.y                                             # labels: used ONLY to describe errors, never for training

def get_scores(model, pairs):
    model.eval()
    with torch.no_grad():
        z = model.encoder(x, train_mp)
        return torch.sigmoid(model.score(z, pairs.to(device))).cpu().numpy()

def describe(pairs, scores, idx):
    out = []
    for i in idx:
        u, v = pairs[0][i].item(), pairs[1][i].item()
        out.append(dict(u=u, v=v, score=round(float(scores[i]), 3),
                        deg_u=int(deg[u]), deg_v=int(deg[v]),
                        common_neighbours=int(CN_train[u, v]),
                        same_topic=bool(y_cls[u] == y_cls[v])))
    return pd.DataFrame(out)

pos_scores = get_scores(model, test_pos)
neg_scores = get_scores(model, test_neg)

print("Missed true links (lowest-scored test positives):")
print(describe(test_pos, pos_scores, np.argsort(pos_scores)[:5]).to_string(index=False))
print("\nFalse alarms (highest-scored test negatives):")
print(describe(test_neg, neg_scores, np.argsort(-neg_scores)[:5]).to_string(index=False))

Missed true links (lowest-scored test positives):
   u    v  score  deg_u  deg_v  common_neighbours  same_topic
  43 2400  0.000      2      0                  0        True
1693 2408  0.045      1      0                  0        True
 460 1989  0.070      1      0                  0       False
  23 2159  0.095      0      2                  0        True
1058 1195  0.168      0      1                  0        True

False alarms (highest-scored test negatives):
   u    v  score  deg_u  deg_v  common_neighbours  same_topic
1240 2130    1.0      5     10                  0        True
 306  392    1.0     51      5                  0       False
 118  861    1.0     15      4                  0       False
 118 1637    1.0     15      2                  0       False
1701 2191    1.0     53      0                  0        True


##Ten candidate links

In [19]:
full_mp = both_directions(unique_pos).to(device)
model.eval()
with torch.no_grad():
    z = model.encoder(x, full_mp)
    S = torch.zeros(N, N, device=device)
    for i in range(0, N, 100):
        S[i:i + 100] = torch.sigmoid(model.decoder(z[i:i + 100].unsqueeze(1), z.unsqueeze(0)))

observed = torch.zeros(N, N, dtype=torch.bool, device=device)
observed[full_mp[0], full_mp[1]] = True
allowed = torch.triu(torch.ones(N, N, dtype=torch.bool, device=device), diagonal=1) & ~observed
S[~allowed] = -1                                           # drop self-pairs, duplicates and observed links

vals, idx = S.view(-1).topk(10)
CN_full = common_neighbor_matrix(full_mp)
deg_full = torch.bincount(full_mp[0].cpu(), minlength=N)

cand = []
for val, i in zip(vals.tolist(), idx.tolist()):
    u, v = i // N, i % N
    cand.append(dict(u=u, v=v, score=round(val, 3), common_neighbours=int(CN_full[u, v]),
                     deg_u=int(deg_full[u]), deg_v=int(deg_full[v]), same_topic=bool(y_cls[u] == y_cls[v])))
print(pd.DataFrame(cand).to_string(index=False))

 u    v  score  common_neighbours  deg_u  deg_v  same_topic
 2 1701    1.0                  0      5     74       False
 0 1701    1.0                  2      3     74        True
 2 1810    1.0                  0      5     44       False
 0 1810    1.0                  0      3     44        True
 0 1986    1.0                  0      3     65        True
 2 2034    1.0                  0      5     40        True
 7 1701    1.0                  0      1     74        True
 7 1810    1.0                  0      1     44        True
 1 2034    1.0                  0      3     40        True
 0  598    1.0                  0      3     33        True
